In [1]:
import pandas as pd
import numpy as np

from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, BatchNormalization,LSTM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.preprocessing.sequence import pad_sequences

I0000 00:00:1787591436.693282  108634 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787591436.694605  108634 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787591436.781345  108634 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787591438.976910  108634 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
x_train = np.load(r"../data/cleaned/x_train.npy")
x_test = np.load(r"../data/cleaned/x_test.npy")
x_val = np.load(r"../data/cleaned/x_val.npy")
y_train = np.load(r"../data/cleaned/y_train.npy")
y_test = np.load(r"../data/cleaned/y_test.npy")
y_val = np.load(r"../data/cleaned/y_val.npy")

In [3]:
x_train.shape,x_test.shape,x_val.shape

((34705, 200), (7438, 200), (7439, 200))

In [4]:
y_train.shape,y_test.shape,y_val.shape

((34705,), (7438,), (7439,))

In [5]:
VOCAB_SIZE = 20000
MAX_LEN = 200

In [ ]:
'''Embedding'''

In [6]:
lstm_model_100 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=100
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

E0000 00:00:1787591951.001758  108634 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
lstm_model_100.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_100.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 107s 96ms/step - accuracy: 0.5159 - loss: 0.6897 - val_accuracy: 0.5257 - val_loss: 0.6784
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 103s 95ms/step - accuracy: 0.6137 - loss: 0.6346 - val_accuracy: 0.7238 - val_loss: 0.5664
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 104s 95ms/step - accuracy: 0.7879 - loss: 0.4483 - val_accuracy: 0.8664 - val_loss: 0.3182
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 118s 108ms/step - accuracy: 0.9193 - loss: 0.2161 - val_accuracy: 0.8845 - val_loss: 0.2944
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.9546 - loss: 0.1371 - val_accuracy: 0.8730 - val_loss: 0.3570
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 107s 99ms/step - accuracy: 0.9735 - loss: 0.0869 - val_accuracy: 0.8767 - val_loss: 0.3720
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 114s 105ms/step - accuracy: 0.9862 - loss: 0.0509 - val_accuracy: 0.8722 - val_loss: 0.4759
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 110s 102ms/step - accura

In [8]:
y_pred = lstm_model_100.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step


In [9]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8646323430568624
Precision: 0.8813654168998322
Recall   : 0.8438253415483525
F1 Score : 0.8621869440262762
auc_score : 0.8647081375847537


In [10]:
lstm_model_100.save(r"../models/lstm_model_em100.keras")

history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/lstm_em100.csv',index=False)

In [ ]:
'''50'''

In [11]:
lstm_model_em50 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=50
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [12]:
lstm_model_em50.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_em50.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 101s 92ms/step - accuracy: 0.5148 - loss: 0.6911 - val_accuracy: 0.5227 - val_loss: 0.6851
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 98s 90ms/step - accuracy: 0.5940 - loss: 0.6367 - val_accuracy: 0.5415 - val_loss: 0.6698
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 97s 90ms/step - accuracy: 0.7516 - loss: 0.5156 - val_accuracy: 0.7845 - val_loss: 0.5312
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 104s 96ms/step - accuracy: 0.8241 - loss: 0.4417 - val_accuracy: 0.7633 - val_loss: 0.5575
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 64s 59ms/step - accuracy: 0.8400 - loss: 0.3801 - val_accuracy: 0.8519 - val_loss: 0.3938
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 67ms/step - accuracy: 0.9050 - loss: 0.2469 - val_accuracy: 0.8672 - val_loss: 0.3482
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 110s 101ms/step - accuracy: 0.9423 - loss: 0.1617 - val_accuracy: 0.8754 - val_loss: 0.3910
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 114s 105ms/step - accuracy: 0.

In [13]:
y_pred = lstm_model_em50.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step


In [14]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8693372765156607
Precision: 0.8666666666666667
Recall   : 0.8740959014197697
F1 Score : 0.8703654307815417
auc_score : 0.8693199420752383


In [15]:
lstm_model_em50.save(r"../models/lstm_model_em50.keras")

history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/lstm_em50.csv',index=False)

In [ ]:
'''128'''

In [16]:
lstm_model_em128 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [17]:
lstm_model_em128.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_em128.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 76ms/step - accuracy: 0.5172 - loss: 0.6895 - val_accuracy: 0.5287 - val_loss: 0.6827
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 77ms/step - accuracy: 0.6258 - loss: 0.6081 - val_accuracy: 0.8282 - val_loss: 0.4180
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.8877 - loss: 0.2834 - val_accuracy: 0.8887 - val_loss: 0.2840
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9503 - loss: 0.1457 - val_accuracy: 0.8847 - val_loss: 0.3035
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.9789 - loss: 0.0711 - val_accuracy: 0.8750 - val_loss: 0.4105
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9916 - loss: 0.0334 - val_accuracy: 0.8763 - val_loss: 0.4749
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9948 - loss: 0.0213 - val_accuracy: 0.8736 - val_loss: 0.5819
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.9964 -

In [18]:
y_pred = lstm_model_em128.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step


In [19]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8737733566339562
Precision: 0.8717402873869079
Recall   : 0.8775783552102866
F1 Score : 0.8746495794953945
auc_score : 0.8737594960077336


In [20]:
lstm_model_em128.save(r"../models/lstm_model_em128.keras")

history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/lstm_em128.csv',index=False)

In [ ]:
'''hidden unit'''

In [21]:
lstm_model_un32 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(32),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [22]:
lstm_model_un32.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_un32.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 71s 64ms/step - accuracy: 0.5280 - loss: 0.6815 - val_accuracy: 0.5471 - val_loss: 0.6674
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 69ms/step - accuracy: 0.7062 - loss: 0.5144 - val_accuracy: 0.8493 - val_loss: 0.3733
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 67s 62ms/step - accuracy: 0.9125 - loss: 0.2307 - val_accuracy: 0.8882 - val_loss: 0.2837
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 67s 62ms/step - accuracy: 0.9617 - loss: 0.1145 - val_accuracy: 0.8699 - val_loss: 0.3716
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 91s 84ms/step - accuracy: 0.9810 - loss: 0.0635 - val_accuracy: 0.8722 - val_loss: 0.3989
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 71s 65ms/step - accuracy: 0.9846 - loss: 0.0503 - val_accuracy: 0.8615 - val_loss: 0.4792
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 69s 64ms/step - accuracy: 0.9905 - loss: 0.0328 - val_accuracy: 0.8664 - val_loss: 0.5580
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 69s 63ms/step - accuracy: 0.9965 -

In [23]:
y_pred = lstm_model_un32.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step


In [24]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8710848232289287
Precision: 0.8724489795918368
Recall   : 0.8703455665684436
F1 Score : 0.8713960037548613
auc_score : 0.8710875161498451


In [ ]:
'''128'''

In [25]:
lstm_model_un128 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [26]:
lstm_model_un128.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_un128.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 133s 121ms/step - accuracy: 0.5180 - loss: 0.6908 - val_accuracy: 0.5333 - val_loss: 0.6817
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 130s 120ms/step - accuracy: 0.6796 - loss: 0.5672 - val_accuracy: 0.8078 - val_loss: 0.4069
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 129s 119ms/step - accuracy: 0.8941 - loss: 0.2652 - val_accuracy: 0.8841 - val_loss: 0.2765
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 128s 118ms/step - accuracy: 0.9522 - loss: 0.1380 - val_accuracy: 0.8836 - val_loss: 0.3070
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 130s 120ms/step - accuracy: 0.9803 - loss: 0.0662 - val_accuracy: 0.8754 - val_loss: 0.4019
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 121s 111ms/step - accuracy: 0.9914 - loss: 0.0351 - val_accuracy: 0.8744 - val_loss: 0.5107
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 125s 115ms/step - accuracy: 0.9949 - loss: 0.0209 - val_accuracy: 0.8714 - val_loss: 0.5382
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 132s 122ms/step - ac

In [27]:
y_pred = lstm_model_un128.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step


In [28]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.87121924989918
Precision: 0.8658054310572106
Recall   : 0.8797214036967587
F1 Score : 0.8727079457879352
auc_score : 0.8711882787506999


In [ ]:
'''drop out'''

In [29]:
lstm_model_drop3 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

In [30]:
lstm_model_drop3.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_drop3.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 78ms/step - accuracy: 0.5207 - loss: 0.6859 - val_accuracy: 0.5220 - val_loss: 0.6870
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.6005 - loss: 0.6234 - val_accuracy: 0.8363 - val_loss: 0.4079
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 85s 79ms/step - accuracy: 0.8779 - loss: 0.3105 - val_accuracy: 0.8681 - val_loss: 0.3120
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.9388 - loss: 0.1792 - val_accuracy: 0.8874 - val_loss: 0.3013
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.9703 - loss: 0.1018 - val_accuracy: 0.8821 - val_loss: 0.3660
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - accuracy: 0.9841 - loss: 0.0623 - val_accuracy: 0.8770 - val_loss: 0.4265
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 81s 75ms/step - accuracy: 0.9894 - loss: 0.0417 - val_accuracy: 0.8743 - val_loss: 0.4990
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 92s 85ms/step - accuracy: 0.9930 -

In [31]:
y_pred = lstm_model_drop3.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step


In [32]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8611372496303267
Precision: 0.8281477880408362
Recall   : 0.9126707741762657
F1 Score : 0.8683573340129986
auc_score : 0.8609495263218079


In [ ]:
#drop out 5

In [33]:
lstm_model_drop5 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

In [34]:
lstm_model_drop5.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_drop5.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 76ms/step - accuracy: 0.5092 - loss: 0.6921 - val_accuracy: 0.5264 - val_loss: 0.6880
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 66ms/step - accuracy: 0.6873 - loss: 0.5469 - val_accuracy: 0.8583 - val_loss: 0.3451
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 67s 62ms/step - accuracy: 0.9036 - loss: 0.2612 - val_accuracy: 0.8753 - val_loss: 0.2979
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 70ms/step - accuracy: 0.9511 - loss: 0.1483 - val_accuracy: 0.8876 - val_loss: 0.3219
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 76s 70ms/step - accuracy: 0.9781 - loss: 0.0747 - val_accuracy: 0.8790 - val_loss: 0.4018
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 87s 80ms/step - accuracy: 0.9880 - loss: 0.0453 - val_accuracy: 0.8781 - val_loss: 0.5196
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.9929 - loss: 0.0271 - val_accuracy: 0.8742 - val_loss: 0.5434
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9957 -

In [35]:
y_pred = lstm_model_drop5.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step


In [36]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8710848232289287
Precision: 0.8746623446785521
Recall   : 0.8673988748995446
F1 Score : 0.8710154673839946
auc_score : 0.8710982501858758


In [ ]:
#recurrent drop out

In [37]:
lstm_model_re_3 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64,
    dropout =0.3,
    recurrent_dropout=0.2),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [38]:
lstm_model_re_3.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_re_3.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 127s 114ms/step - accuracy: 0.5155 - loss: 0.6900 - val_accuracy: 0.5268 - val_loss: 0.6787
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 129s 118ms/step - accuracy: 0.5477 - loss: 0.6580 - val_accuracy: 0.5489 - val_loss: 0.6652
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 132ms/step - accuracy: 0.8321 - loss: 0.3829 - val_accuracy: 0.8554 - val_loss: 0.3347
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 135ms/step - accuracy: 0.9160 - loss: 0.2278 - val_accuracy: 0.8747 - val_loss: 0.3205
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 141s 130ms/step - accuracy: 0.9466 - loss: 0.1524 - val_accuracy: 0.8755 - val_loss: 0.3335
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 138ms/step - accuracy: 0.9687 - loss: 0.0984 - val_accuracy: 0.8731 - val_loss: 0.3891
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 148s 136ms/step - accuracy: 0.9808 - loss: 0.0629 - val_accuracy: 0.8798 - val_loss: 0.4695
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 147s 136ms/step - ac

In [39]:
y_pred = lstm_model_re_3.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step


In [40]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8732356499529507
Precision: 0.8519172552976791
Recall   : 0.9046343423519957
F1 Score : 0.8774847343120696
auc_score : 0.8731212726330945


In [ ]:
#batch normalization

In [41]:
lstm_model_bn = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64,
    dropout =0.3,
    recurrent_dropout=0.2),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [42]:
lstm_model_bn.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_bn.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 132ms/step - accuracy: 0.5124 - loss: 0.6934 - val_accuracy: 0.5514 - val_loss: 0.8428
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 156s 144ms/step - accuracy: 0.5647 - loss: 0.6667 - val_accuracy: 0.5892 - val_loss: 0.6768
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 151s 139ms/step - accuracy: 0.7513 - loss: 0.4702 - val_accuracy: 0.8812 - val_loss: 0.2875
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 161s 148ms/step - accuracy: 0.9182 - loss: 0.2203 - val_accuracy: 0.8914 - val_loss: 0.2836
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 158s 145ms/step - accuracy: 0.9594 - loss: 0.1291 - val_accuracy: 0.8887 - val_loss: 0.3395
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.9733 - loss: 0.0860 - val_accuracy: 0.8781 - val_loss: 0.4428
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 134ms/step - accuracy: 0.9821 - loss: 0.0608 - val_accuracy: 0.8771 - val_loss: 0.4409
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 145s 133ms/step - ac

In [43]:
y_pred = lstm_model_bn.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step


In [44]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8744454899852131
Precision: 0.8871369294605809
Recall   : 0.8590945620144655
F1 Score : 0.8728905824714208
auc_score : 0.8745014094475997


In [45]:
lstm_model_bn1 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [46]:
lstm_model_bn1.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_bn1.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 66ms/step - accuracy: 0.5196 - loss: 0.6863 - val_accuracy: 0.4983 - val_loss: 3.6178
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 77s 71ms/step - accuracy: 0.8370 - loss: 0.3458 - val_accuracy: 0.8778 - val_loss: 0.3156
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 71s 65ms/step - accuracy: 0.9337 - loss: 0.1731 - val_accuracy: 0.8790 - val_loss: 0.2872
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 80s 74ms/step - accuracy: 0.9684 - loss: 0.0910 - val_accuracy: 0.8708 - val_loss: 0.4025
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - accuracy: 0.9859 - loss: 0.0440 - val_accuracy: 0.8723 - val_loss: 0.4313
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 84s 78ms/step - accuracy: 0.9922 - loss: 0.0232 - val_accuracy: 0.8685 - val_loss: 0.5964
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 81s 75ms/step - accuracy: 0.9934 - loss: 0.0200 - val_accuracy: 0.8715 - val_loss: 0.6258
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 80s 73ms/step - accuracy: 0.9959 -

In [47]:
y_pred = lstm_model_bn1.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step


In [48]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8608683962898239
Precision: 0.902206320810972
Recall   : 0.8106080900080365
F1 Score : 0.8539579511782136
auc_score : 0.8610514815933328


In [ ]:
#OPTIMIZER

In [49]:
lstm_model_RMS = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [50]:
lstm_model_RMS.compile(optimizer="RMSprop",loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_RMS.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 65ms/step - accuracy: 0.5815 - loss: 0.6585 - val_accuracy: 0.5018 - val_loss: 0.8186
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 66ms/step - accuracy: 0.7636 - loss: 0.5097 - val_accuracy: 0.6947 - val_loss: 0.8324
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 72s 66ms/step - accuracy: 0.8579 - loss: 0.3509 - val_accuracy: 0.8696 - val_loss: 0.3153
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 76s 70ms/step - accuracy: 0.9067 - loss: 0.2400 - val_accuracy: 0.8926 - val_loss: 0.2625
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.9354 - loss: 0.1806 - val_accuracy: 0.8662 - val_loss: 0.3495
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 74s 68ms/step - accuracy: 0.9553 - loss: 0.1324 - val_accuracy: 0.8843 - val_loss: 0.3428
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 74s 69ms/step - accuracy: 0.9724 - loss: 0.0929 - val_accuracy: 0.8779 - val_loss: 0.3991
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 70ms/step - accuracy: 0.9825 -

In [51]:
y_pred = lstm_model_RMS.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step


In [52]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.869471703185912
Precision: 0.8979827089337176
Recall   : 0.8347173854808465
F1 Score : 0.8651950576148827
auc_score : 0.8695983041813299


In [ ]:
#learning rate tuning

In [53]:
lstm_model_lr1 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [55]:
lstm_model_lr1.compile(optimizer=optimizers.Adam(learning_rate=0.001),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_lr1.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 82s 74ms/step - accuracy: 0.5434 - loss: 0.6807 - val_accuracy: 0.6298 - val_loss: 0.6547
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.6680 - loss: 0.6193 - val_accuracy: 0.5889 - val_loss: 0.6887
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 77s 71ms/step - accuracy: 0.8809 - loss: 0.2897 - val_accuracy: 0.8825 - val_loss: 0.2836
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 77s 71ms/step - accuracy: 0.9427 - loss: 0.1575 - val_accuracy: 0.8852 - val_loss: 0.3082
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9738 - loss: 0.0828 - val_accuracy: 0.8696 - val_loss: 0.4140
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 98s 91ms/step - accuracy: 0.9846 - loss: 0.0492 - val_accuracy: 0.8676 - val_loss: 0.5121
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 95s 87ms/step - accuracy: 0.9899 - loss: 0.0335 - val_accuracy: 0.8683 - val_loss: 0.5701
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 94s 87ms/step - accuracy: 0.9933 -

In [56]:
y_pred = lstm_model_lr1.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step


In [57]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8661110364296276
Precision: 0.8658112804063085
Recall   : 0.8676667559603536
F1 Score : 0.8667380251538668
auc_score : 0.8661053693455304


In [ ]:
#lr 2

In [58]:
lstm_model_lr2 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [59]:
lstm_model_lr2.compile(optimizer=optimizers.Adam(learning_rate=0.0005),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_lr2.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 77s 69ms/step - accuracy: 0.5886 - loss: 0.6546 - val_accuracy: 0.4982 - val_loss: 0.7070
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 91s 84ms/step - accuracy: 0.6377 - loss: 0.6286 - val_accuracy: 0.5085 - val_loss: 0.7838
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 98s 91ms/step - accuracy: 0.7937 - loss: 0.4098 - val_accuracy: 0.8857 - val_loss: 0.2800
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 93s 86ms/step - accuracy: 0.9289 - loss: 0.1879 - val_accuracy: 0.8630 - val_loss: 0.3286
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 92s 85ms/step - accuracy: 0.9584 - loss: 0.1197 - val_accuracy: 0.8801 - val_loss: 0.3417
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 94s 87ms/step - accuracy: 0.9781 - loss: 0.0662 - val_accuracy: 0.8754 - val_loss: 0.4078
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 98s 91ms/step - accuracy: 0.9871 - loss: 0.0407 - val_accuracy: 0.8412 - val_loss: 0.7431
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.9913 -

In [60]:
y_pred = lstm_model_lr2.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step


In [61]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8724290899314424
Precision: 0.8657383079348397
Recall   : 0.8826680953656576
F1 Score : 0.8741212362382279
auc_score : 0.8723917918814257


In [ ]:
#lr3

In [62]:
lstm_model_lr3 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [63]:
lstm_model_lr3.compile(optimizer=optimizers.Adam(learning_rate=0.0001),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_lr3.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 66ms/step - accuracy: 0.5101 - loss: 0.6915 - val_accuracy: 0.5288 - val_loss: 0.6844
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 79s 73ms/step - accuracy: 0.7671 - loss: 0.4362 - val_accuracy: 0.8689 - val_loss: 0.3361
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 91s 83ms/step - accuracy: 0.9118 - loss: 0.2294 - val_accuracy: 0.8917 - val_loss: 0.2741
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 93s 86ms/step - accuracy: 0.9350 - loss: 0.1778 - val_accuracy: 0.8813 - val_loss: 0.3556
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 96s 88ms/step - accuracy: 0.9476 - loss: 0.1497 - val_accuracy: 0.8607 - val_loss: 0.4958
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 94s 86ms/step - accuracy: 0.9603 - loss: 0.1227 - val_accuracy: 0.8623 - val_loss: 0.4116
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 95s 88ms/step - accuracy: 0.9636 - loss: 0.1183 - val_accuracy: 0.8361 - val_loss: 0.8144
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 96s 89ms/step - accuracy: 0.9733 -

In [64]:
y_pred = lstm_model_lr3.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step


In [65]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8686651431644038
Precision: 0.8611111111111112
Recall   : 0.8802571658183767
F1 Score : 0.8705788846204795
auc_score : 0.8686229164224101


In [ ]:
#batch size

In [66]:
lstm_model_64 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [67]:
lstm_model_64.compile(optimizer=optimizers.Adam(),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_64.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=64)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 60s 106ms/step - accuracy: 0.5662 - loss: 0.6636 - val_accuracy: 0.5064 - val_loss: 0.6880
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 54s 99ms/step - accuracy: 0.7164 - loss: 0.4849 - val_accuracy: 0.7375 - val_loss: 0.6640
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 53s 98ms/step - accuracy: 0.9251 - loss: 0.2021 - val_accuracy: 0.8227 - val_loss: 0.4277
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 54s 100ms/step - accuracy: 0.9672 - loss: 0.0996 - val_accuracy: 0.8369 - val_loss: 0.4916
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 57s 104ms/step - accuracy: 0.9838 - loss: 0.0503 - val_accuracy: 0.8736 - val_loss: 0.6005
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 56s 102ms/step - accuracy: 0.9908 - loss: 0.0306 - val_accuracy: 0.8763 - val_loss: 0.5557
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 58s 106ms/step - accuracy: 0.9910 - loss: 0.0276 - val_accuracy: 0.8560 - val_loss: 0.6272
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 55s 100ms/step - accuracy: 0.9915 - loss: 0.0

In [68]:
y_pred = lstm_model_64.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step


In [69]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8654389030783708
Precision: 0.8482916879143294
Recall   : 0.8912402893115456
F1 Score : 0.869235793598955
auc_score : 0.8653449152979746


In [ ]:
#eraly stopping

In [70]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [71]:
lstm_model_lr = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [73]:
lstm_model_lr.compile(optimizer=optimizers.Adam(),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_lr.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32, callbacks=[early_stop])

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 87s 78ms/step - accuracy: 0.5348 - loss: 0.6778 - val_accuracy: 0.4983 - val_loss: 0.7737
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 85s 79ms/step - accuracy: 0.7333 - loss: 0.4703 - val_accuracy: 0.8024 - val_loss: 0.4004
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 88s 81ms/step - accuracy: 0.8990 - loss: 0.2473 - val_accuracy: 0.8829 - val_loss: 0.2944
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 92s 85ms/step - accuracy: 0.9582 - loss: 0.1252 - val_accuracy: 0.8812 - val_loss: 0.3654
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 96s 88ms/step - accuracy: 0.9796 - loss: 0.0676 - val_accuracy: 0.8781 - val_loss: 0.4174
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 89s 82ms/step - accuracy: 0.9894 - loss: 0.0405 - val_accuracy: 0.8714 - val_loss: 0.5412
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [74]:
y_pred = lstm_model_lr.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step


In [75]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8829143702110499
Precision: 0.9164726426076834
Recall   : 0.8435574604875435
F1 Score : 0.8785046728971962
auc_score : 0.883057737259422


In [ ]:
#leraning sheduler

In [76]:
lstm_model_lrschdelur = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [77]:
lstm_model_lrschdelur.compile(optimizer=optimizers.Adam(),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_lrschdelur.fit(x_train,y_train, validation_data=(x_val,y_val),epochs=10,batch_size=32, callbacks=[lr_scheduler])

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 78s 69ms/step - accuracy: 0.6358 - loss: 0.5722 - val_accuracy: 0.6170 - val_loss: 1.1944 - learning_rate: 0.0010
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 76s 70ms/step - accuracy: 0.9057 - loss: 0.2388 - val_accuracy: 0.8256 - val_loss: 0.4631 - learning_rate: 0.0010
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 69ms/step - accuracy: 0.9580 - loss: 0.1224 - val_accuracy: 0.8106 - val_loss: 0.6013 - learning_rate: 0.0010
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 76s 70ms/step - accuracy: 0.9827 - loss: 0.0556 - val_accuracy: 0.8800 - val_loss: 0.4851 - learning_rate: 0.0010
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 79s 73ms/step - accuracy: 0.9933 - loss: 0.0246 - val_accuracy: 0.8789 - val_loss: 0.5102 - learning_rate: 5.0000e-04
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 80s 73ms/step - accuracy: 0.9978 - loss: 0.0106 - val_accuracy: 0.8789 - val_loss: 0.6584 - learning_rate: 5.0000e-04
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 75s 69ms/step 

In [83]:
y_pred = lstm_model_lrschdelur.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step


In [84]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8755209033472241
Precision: 0.8733705772811918
Recall   : 0.8794535226359497
F1 Score : 0.8764014949279231
auc_score : 0.8755065778317362


In [ ]:
#seq tuning

In [78]:
x_trainseq = np.load(r"../data/seq/x_train_seq.npy", allow_pickle=True)
x_testseq = np.load(r"../data/seq/x_test_seq.npy", allow_pickle=True)
x_valseq = np.load(r"../data/seq/x_val_seq.npy", allow_pickle=True)

In [79]:
def create_sequences(length):
    x_train_pad = pad_sequences(
        x_trainseq,
        maxlen=length,
        padding="post",
        truncating="post"
    )

    x_val_pad = pad_sequences(
        x_valseq,
        maxlen=length,
        padding="post",
        truncating="post"
    )

    x_test_pad = pad_sequences(
        x_testseq,
        maxlen=length,
        padding="post",
        truncating="post"
    )
    return x_train_pad, x_val_pad, x_test_pad

In [80]:
x_train_300, x_val_300, x_test_300 = create_sequences(300)

In [81]:
lstm_model_seq300 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [82]:
lstm_model_seq300.compile(optimizer=optimizers.Adam(),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_seq300.fit(x_train_300,y_train, validation_data=(x_val_300,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 114s 103ms/step - accuracy: 0.5212 - loss: 0.6894 - val_accuracy: 0.5018 - val_loss: 0.7222
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.6634 - loss: 0.5569 - val_accuracy: 0.8775 - val_loss: 0.2942
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.9095 - loss: 0.2293 - val_accuracy: 0.8810 - val_loss: 0.3043
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 120s 111ms/step - accuracy: 0.9584 - loss: 0.1200 - val_accuracy: 0.8844 - val_loss: 0.3219
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 118s 109ms/step - accuracy: 0.9830 - loss: 0.0571 - val_accuracy: 0.8861 - val_loss: 0.4185
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 102s 94ms/step - accuracy: 0.9910 - loss: 0.0324 - val_accuracy: 0.8761 - val_loss: 0.4866
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 113s 104ms/step - accuracy: 0.9944 - loss: 0.0196 - val_accuracy: 0.8794 - val_loss: 0.6320
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 121s 111ms/step - acc

In [85]:
y_pred = lstm_model_seq300.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step


In [86]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.876193036698481
Precision: 0.8877551020408163
Recall   : 0.8623091347441736
F1 Score : 0.8748471259682022
auc_score : 0.8762436121643156


In [87]:
lstm_model_seq300.save(r"../models/rnn_model_re2.keras")

history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/lstm_seq300.csv',index=False)

In [ ]:
#100

In [88]:
x_train_100, x_val_100, x_test_100 = create_sequences(100)

In [ ]:
lstm_model_seq100 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    LSTM(64),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

In [90]:
lstm_model_seq100.compile(optimizer=optimizers.Adam(),loss="binary_crossentropy",metrics=["accuracy"])
history = lstm_model_seq100.fit(x_train_100,y_train, validation_data=(x_val_100,y_val),epochs=10,batch_size=32)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 63s 56ms/step - accuracy: 0.7988 - loss: 0.4228 - val_accuracy: 0.8623 - val_loss: 0.3470
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 49ms/step - accuracy: 0.9037 - loss: 0.2499 - val_accuracy: 0.8658 - val_loss: 0.3226
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 44s 41ms/step - accuracy: 0.9267 - loss: 0.2027 - val_accuracy: 0.8540 - val_loss: 0.4114
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 60s 55ms/step - accuracy: 0.9480 - loss: 0.1469 - val_accuracy: 0.7685 - val_loss: 0.6147
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 56s 51ms/step - accuracy: 0.9529 - loss: 0.1330 - val_accuracy: 0.8406 - val_loss: 0.5723
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 51s 47ms/step - accuracy: 0.9742 - loss: 0.0792 - val_accuracy: 0.8447 - val_loss: 0.5831
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 51s 47ms/step - accuracy: 0.9856 - loss: 0.0510 - val_accuracy: 0.7869 - val_loss: 0.8396
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 47s 44ms/step - accuracy: 0.9868 -

In [91]:
y_pred = lstm_model_seq100.predict(x_val)
y_pred = (y_pred >= 0.5).astype(int).ravel()

233/233 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step


In [92]:
accuracy_lstm = accuracy_score(y_val, y_pred)
precision_lstm = precision_score(y_val, y_pred)
recall_lstm = recall_score(y_val, y_pred)
f1_lstm = f1_score(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

print("Accuracy :", accuracy_lstm)
print("Precision:", precision_lstm)
print("Recall   :", recall_lstm)
print("F1 Score :", f1_lstm)
print("auc_score :",auc_score)

Accuracy : 0.8628847963435946
Precision: 0.8585778482685699
Recall   : 0.8700776855076346
F1 Score : 0.8642895156998404
auc_score : 0.8628585945077298
